# Essential & Vital Workers — Workflow Notebook

This notebook is the **guided walkthrough** for the essential-worker model.

Use this notebook to:

1. Reproduce every CSV under `results/essential_workers/` from `data/essential_workers/` inputs.
2. See each stage step by step, not only the one-shot `ew.estimate()`.
3. Compare the different **indoor-fraction** methods.
4. Inspect overlap calibration, on-site housing adjustments, and ILO validation.

**Downstream:** `src/filtration_scale_up_model.py`, `scripts/filtration_scale_up_walkthrough.ipynb`
and the figure scripts read `results/essential_workers/essential_workers_by_country.csv`.

### Inputs (`data/essential_workers/`)

| File | Role |
| --- | --- |
| `parameters_essential_workers.csv` | Every setting: indoor method, group overlaps, pathogen, masks, airflow credit |
| `isco08_opinion_poll_census.xlsx` | In-house vital poll (0/1) at ISCO L4 |
| `onet_indoors_environmentally_controlled.csv` | O*NET indoor context % (controlled) |
| `onet_indoors_not_environmentally_controlled.csv` | O*NET indoor context % (not controlled) |
| `isco_soc_crosswalk.csv` | SOC → ISCO-08 mapping |
| `ilo_isco08_employment.csv` | ILO employment by country × ISCO L2 |
| `labour_force_wb_plus.xlsx` | World Bank labour force 2024 |
| `ilo_country_essential_workers_pct.xlsx` | ILO WESO 2023 published %essential |
| `job_exposure_matrix.xls` | JEM Location for `jem_partial` / `jem_binary` |
| `similar_countries.csv` | Similar countries used to fill countries without ILO data |
| `ashrae241_group_mapping.csv`, `ashrae241_eca_by_occupancy.csv` | ASHRAE-241 room type per group |
| `pathogen_viral_loads.csv` | Viral loads used to scale ASHRAE-241 to other pathogens |

### Outputs (`results/essential_workers/`)

| File | Contents |
| --- | --- |
| `essential_workers_by_country.csv` | Per-country counts, percentages and eCADR requirements |
| `essential_workers_by_group.csv` | Per-country × occupational-group counts and requirements |
| `essential_workers_by_region.csv` | UN regional aggregates |
| `onsite_housing_worker_requirements.csv` | Housing-relevant totals (excl. ISCO 61+63) |
| `group_composition_global.csv`, `group_composition_by_region.csv` | Occupational make-up of each workforce |
| `ashrae241_scaled_table1.csv`, `ashrae241_scaled_by_mask_efficiency.csv` | ASHRAE-241 Table 1 scaled to the pathogen |
| `validation/` | ILO validation, overlap calibration, indoor-method sensitivity and GDP correlations |


## Methodology Overview

> ILO (2023). **World Employment and Social Outlook 2023: The value of essential work.**
> International Labour Organization.
> [WCMS_871016](https://www.ilo.org/sites/default/files/wcmsp5/groups/public/@dgreports/@dcomm/@publ/documents/publication/wcms_871016.pdf)

The ILO classifies a worker as a *key worker* (essential worker) if **both** of these are true:

1. They are in a key **occupation** (ISCO-08 code listed in Table A2 of the report).
2. They are in a key **industry** (ISIC Rev.4 code listed in Table A1 of the report).

So `essential = occupation AND industry`, not just one or the other. The ILO computes its per-country shares from worker-level microdata in which every respondent has both an ISCO code and an ISIC code, so the intersection is exact.

### Our calculations

**We do not have access to ILO worker-level ISCO x ISIC microdata.** The only country-level breakdown we have is ILO employment per ISCO-08 L2 code (`ilo_isco08_employment.csv`). To approximate the intersection we use Figure A1 from the linked report above as **global priors** (the `group_overlap_*` rows of `parameters_essential_workers.csv`) and then **calibrate** them per country.

**1. Global priors.** For each occupational group *g*, the overlap is the globally aggregated fraction of workers in group *g* that are also in a key ISIC industry (ILO Figure A1). Armed Forces uses 0.40 (Blueprint; ILO excludes uniformed services from headline figures).

The global average group overlap factors:

| Group | Overlap | Source |
| --- | ---: | --- |
| Food | 0.895 | ILO Figure A1 |
| Health | 0.819 | ILO Figure A1 |
| Retail | 0.876 | ILO Figure A1 |
| Security | 0.846 | ILO Figure A1 |
| Transport | 0.869 | ILO Figure A1 |
| Manual | 0.335 | ILO Figure A1 |
| Cleaning | 0.485 | ILO Figure A1 |
| Tech | 0.320 | ILO Figure A1 |
| Armed Forces | **0.40** | Blueprint Biosecurity's "A theory of pandemic-proof PPE" https://blueprintbiosecurity.org/u/2024/05/BB_Next-Gen-Report_PRF9-WEB-1.pdf?utm_source=bluedot-impact (as the ILO excludes armed forces from its global figures) |

**2. Per-country calibration.** For each country with ILO ISCO employment and a published WESO %essential, one scalar `x ∈ [0, 1]` moves all eight calibratable groups together: toward 1.0 when the model under-shoots ILO, toward 0 when it over-shoots. Armed Forces overlap stays fixed at 0.40. This is implemented in `essential_workers.calibrate_group_overlaps` and logged in `results/essential_workers/validation/group_overlap_calibration.csv`. For countries without data (e.g. China), calibrated overlaps are the mean of the similar countries' calibrated values (`data/essential_workers/similar_countries.csv`).

**3. Applying overlaps to calculate vital workers** using the calibrated overlaps obtained from step 2, we apply them to a smaller set of ISCO-08 job categories. These were defined by team members each flagging ISCO-08 job categories as vital or not vital at the 4-digit code level (specific job categories). As the ILO employment data is at the 2-digit level (broader job categories), we take the mean of the 4-digit codes within each 2-digit category.


**4. Estimate indoor workers** We multiply the essential worker weights and vital worker weights by `indoors_context` (0–1), which is the fraction of that job category spent inside
and therefore benefitting from in-room air filtration. We derive this from O*NET or JEM datasets on time spent working indoors by job category. The active rule is set by `indoor_context_method` in `data/essential_workers/parameters_essential_workers.csv` (default: `jem_binary`).

**5. Filtration requirements (eCADR).** Indoor essential and indoor vital workers in each occupational group are multiplied by an equivalent clean airflow scaled from ASHRAE-241 up to measles. Section 6 walks through that calculation and the room type assigned to each group.


## 0. Setup


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

import essential_workers as ew
import essential_workers_validation as validation
import processing.preprocessing as pp
from processing.paths import (
    ESSENTIAL_WORKERS_DATA as DATA,
    ESSENTIAL_WORKERS_PARAMETERS,
    ESSENTIAL_WORKERS_RESULTS as RESULTS,
    read_parameters,
)

parameters, _ = read_parameters(ESSENTIAL_WORKERS_PARAMETERS)
JEM_PATH = DATA / 'job_exposure_matrix.xls'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

print(f'Data:    {DATA}')
print(f'Results: {RESULTS}')
print(f"Indoor context method: {parameters['indoor_context_method']}")


## 1. Quick-start — full model

`ew.estimate()` runs the whole model, and `ew.write_results` writes the core
tables, as `python src/essential_workers.py` does. The indoor fraction comes
from `indoor_context_method` in `parameters_essential_workers.csv` (default:
`jem_binary`); pass `indoor_context_method=...` to override it.

`python src/essential_workers_validation.py` writes the validation tables.
Manuscript numbers are printed in **Key results** (section 11).


In [ ]:
results = ew.estimate()
ew.write_results(results)
by_country = results['by_country']
print(f'{len(by_country)} countries and territories.')


### Occupational-group composition

Worker-weighted breakdown of who makes up the essential / vital workforces
(Food, Health, Retail, …). `write_results` saves the global and regional
tables; the figure is from `plot_group_composition.py`.


In [ ]:
composition = ew.group_composition(results['by_group'])
share_columns = [column for column in composition if column.startswith('% of')]
display(composition.set_index('occupational_group')[share_columns].mul(100).round(1))


## 2. Preprocessing — load inputs & build ISCO weights

This section repeats the first half of `estimate` (`processing.preprocessing.read_inputs`):
read the files and build the L2 weight table (poll + indoor context + ILO
essential flag + group overlaps).

### 2.1 Load raw inputs


In [ ]:
poll_df = pd.read_excel(DATA / 'isco08_opinion_poll_census.xlsx')
onet_env_df = pd.read_csv(DATA / 'onet_indoors_environmentally_controlled.csv')
onet_not_df = pd.read_csv(DATA / 'onet_indoors_not_environmentally_controlled.csv')
crosswalk_df = pd.read_csv(DATA / 'isco_soc_crosswalk.csv')
ilo_emp_df = pd.read_csv(DATA / 'ilo_isco08_employment.csv')
lf_raw = pd.read_excel(DATA / 'labour_force_wb_plus.xlsx', usecols=[0, 1, 3])
ilo_pct_df = pp.load_ilo_published_pct(DATA / 'ilo_country_essential_workers_pct.xlsx')

for name, df in [
    ('poll', poll_df), ('ONET env', onet_env_df), ('ONET not', onet_not_df),
    ('crosswalk', crosswalk_df), ('ILO emp', ilo_emp_df), ('LF', lf_raw), ('ILO pct', ilo_pct_df),
]:
    print(f'{name:12s} {df.shape}')


### 2.2 Weight template (before group overlaps)

`build_isco_lvl2_template` attaches `indoors_context` at L4, averages to L2,
sets `Essential Weight ILO`, maps `Group`, and applies the poll rules. The
indoor method is the one in the parameters file.


In [ ]:
def template_for_method(method):
    """ISCO level-2 weights template for one indoor-fraction method."""
    return pp.build_isco_lvl2_template(
        poll_df,
        crosswalk_df,
        onet_controlled_df=onet_env_df,
        onet_not_controlled_df=onet_not_df,
        indoor_context_method=method,
        jem_path=JEM_PATH,
    )


weights_template = template_for_method(parameters['indoor_context_method'])
weights = ew.apply_group_overlaps(weights_template, ew.group_overlaps(parameters))

cols = [
    'Vital Weight POLL', pp.INDOORS_CONTEXT_COLUMN, 'Essential Weight ILO',
    'Group', 'Group Overlap',
    'ISCO_08_PollWeights', 'ISCO_08_ILOWeights',
    'ISCO_08_PollWeights_Total', 'ISCO_08_ILOWeights_Total',
]
display(weights[cols].head(12))

print('Subsistence farmers (63) indoor context:', weights.at['63', pp.INDOORS_CONTEXT_COLUMN])
print('Poll codes the ILO does not class as essential:', pp.NON_ILO_POLL_CODES)


### 2.3 Per-country ILO employment by ISCO L2

`build_employment_by_isco` builds one employment snapshot per country:

- **Sex:** only `sex.label == "Total"` (Male/Female rows are dropped).
- **Year:** one survey year for all ISCO codes in that country — the latest year with NEC (“Not elsewhere classified”) ≤ 10% of total employment; if every year is above 10%, the year with the **lowest** NEC share (ties → more recent year).
- **Headcount:** `obs_value` (thousands) × 1000; ISCO L2 keys are stripped/normalised (`22`, `Tot`, `Not`).
- **Gaps:** missing armed forces and security headcounts take the global median share of coded employment (the US uses DOD end strength), and Laos's missing cleaning codes are filled the same way.

In `ew.worker_shares`, NEC employment is shared out in proportion to the coded
occupations, so each group's count becomes group × (1 + NEC / coded). The
official `Tot` row remains the denominator for % columns.


In [ ]:
employment = pp.build_employment_by_isco(ilo_emp_df)
sample = next(iter(employment))
print(f'Countries with ILO breakdown: {len(employment)}')
display(pd.Series(employment[sample]).head(12))


## 3. Indoor-fraction methods — comparison

The **indoor fraction** (`indoors_context`, 0–1) scales indoor vital/essential counts.
It does **not** change total vital/essential (the `_Total` weight columns omit it).

The model reads `indoor_context_method` from `parameters_essential_workers.csv`
(current default: **`jem_binary`**). This section runs all four methods side by side.

| Method | Source | Rule | When to use |
| --- | --- | --- | --- |
| **`jem_binary`** (default) | `job_exposure_matrix.xls` | JEM Location 0/1 → 0%; 2/3 → 100% | Main results; binary indoor/outdoor from JEM |
| **`jem_partial`** | Same JEM file | 0/1 → 0%; 2 → 50%; 3 → 100% | Sensitivity; treats Location 2 as half-indoor |
| **`onet_max`** | Both O*NET CSVs | Per SOC: `max(env, not_env)` context % ÷ 100 | O*NET-based; smooth 0–1 |
| **`onet_banded`** | Same O*NET | ≥75% → 1.0; 50–75% → 0.5; else 0 | O*NET step-function sensitivity |

All four methods use the same poll, ILO essential flags, and overlap calibration.


### 3.1 L2 `indoors_context` under each method


In [ ]:
indoor_compare = pd.DataFrame({
    method: template_for_method(method)[pp.INDOORS_CONTEXT_COLUMN]
    for method in pp.INDOOR_CONTEXT_METHODS
})
indoor_compare['spread'] = indoor_compare.max(axis=1) - indoor_compare.min(axis=1)
print('L2 codes where methods disagree most (top 10 by spread):')
display(indoor_compare.sort_values('spread', ascending=False).head(10))
print('\nSummary stats of indoors_context by method:')
display(indoor_compare[list(pp.INDOOR_CONTEXT_METHODS)].describe().T)


### 3.2 O*NET SOC → ISCO example (how `onet_max` vs `onet_banded` differ)

At SOC level we take the max of both context columns, then map through the crosswalk.
JEM methods (`jem_partial`, `jem_binary`) use the Location scale instead — see
`preprocessing.location_to_indoor_fraction`.


In [ ]:
soc_merged = pp.merge_onet_max_context(onet_env_df, onet_not_df)
sample_soc = soc_merged.head(5).copy()
for method in ['onet_max', 'onet_banded']:
    sample_soc[method] = sample_soc['context_pct'].apply(
        lambda pct: pp.pct_to_indoor_fraction(pct, method)
    )
display(sample_soc)

print('\nJEM Location bucket examples (mean Location → nearest bucket):')
for location in [0.2, 1.0, 2.1, 2.9]:
    partial = pp.location_to_indoor_fraction(location, partial=True)
    binary = pp.location_to_indoor_fraction(location, partial=False)
    print(f'  Location {location}: jem_partial={partial}, jem_binary={binary}')


### 3.3 Global worker totals — side by side (all four methods)


In [ ]:
sensitivity = validation.indoor_context_sensitivity()
display(
    sensitivity.pivot(
        index='Category', columns='indoor_context_method', values='% of Labour Force'
    ).round(2)
)


## 4. Group overlap calibration

`ew.calibrate_overlaps` does three things:

1. For each country with ILO occupation data and a published ILO share, it
   solves for the scalar `x` that makes essential employment match the ILO.
2. Countries without that data take the mean of their similar countries.
3. Any country still missing takes the global overlaps.

Vital and essential **totals** both use calibrated overlaps; only indoor counts use `indoors_context`.


In [ ]:
calibration = results['calibration']
print('Overlap sources:', calibration['overlap_source'].value_counts().to_dict())
print('\nSample ILO-calibrated countries:')
ilo_rows = calibration[calibration['overlap_source'] == ew.OVERLAP_SOURCE_ILO]
display(ilo_rows.head(3)[[c for c in ilo_rows.columns if 'overlap_' in c or c.startswith('calibration')]])
display(calibration[calibration['Country Code'] == 'AUS'])


In [ ]:
calibration_detail = validation.calibration_detail(results)
print('Mean |adjustment| by group (ILO-calibrated countries):')
ilo_detail = calibration_detail[calibration_detail['Overlap source'] == ew.OVERLAP_SOURCE_ILO]
display(
    ilo_detail.groupby('Group')['Adjustment']
    .apply(lambda s: s.abs().mean())
    .sort_values(ascending=False)
    .to_frame('mean |adjustment|')
)


## 5. Country shares, back-fill, and absolute counts

After calibration, `estimate` adds up each country's group shares, fills
countries without ILO data from similar countries, multiplies by the labour
force, attaches eCADR requirements from each group's ASHRAE-241 room type, and
aggregates to UN regions.

Where the World Bank `Labour Force (2024)` is missing, the model uses ILO `Tot`
employment from the selected country-year snapshot (`ew.fill_missing_labour_force`).


In [ ]:
group_shares, _ = ew.worker_shares(
    results['employment'], results['weights_template'], results['overlaps'],
    ew.group_overlaps(parameters),
)
labour_force = ew.fill_missing_labour_force(pp.prepare_labour_force(lf_raw), employment)
shares = labour_force.merge(ew.country_shares(group_shares), on='Country Name', how='left')
print('Countries without a share before back-fill:', shares['%Essential Workers'].isna().sum())
print('Countries without a share after back-fill: ', by_country['%Essential Workers'].isna().sum())
display(by_country[['Country Name', '%Indoor Essential Workers', '%Essential Workers',
                    ew.LABOUR_FORCE_COL, 'Essential Workers']].head(8))


## 6. Filtration requirements — eCADR scaled to measles

ASHRAE Standard 241 sets an equivalent clean airflow per person (eCADR) for each room type, sized for SARS-CoV-2. `src/ecadr_requirements.py` scales that rate to another pathogen. The default is measles (`ashrae_pathogen`).

1. **Viral load.** log10 viral load is treated as normal. SARS-CoV-2 uses the ASHRAE-241 mean of 7.0 and standard deviation of 1.4. Measles shifts that mean by the log10 of its quanta per ml relative to SARS-CoV-2. Quanta are concentration times the infectivity conversion in `pathogen_viral_loads.csv` (Mikszewski et al. 2022, Table 1). With `viral_load_sd_mode` set to `absolute`, measles uses its own standard deviation from that table.
2. **Quanta emission ratio.** The ratio of the measles viral load to the SARS-CoV-2 viral load at one percentile. `viral_load_percentile` is 96.3, the percentile ASHRAE-241 is designed to.
3. **Masks.** That ratio is multiplied by `(1 - u_new)^2 / (1 - u_base)^2`. `u_base` is the mask efficiency ASHRAE-241 already assumes: 0.30 in health care (`u_base_healthcare`) and 0 elsewhere. `u_new` is the efficiency assumed here: `u_new_healthcare` in health care and `u_new_other` everywhere else.
4. **Scaled eCADR.** Holding infection risk at the ASHRAE-241 level in the Wells-Riley model multiplies the room's eCADR by `k = QER + (QER - 1) * (deposition + decay) / eACH`, where `eACH = eCADR * occupants * 3.6 / room volume` (air changes per hour). Deposition and decay are `deposition_rate_per_hour` and `decay_rate_per_hour`.
5. **Outdoor-air credit.** A share of the room's baseline outdoor airflow (`existing_airflow_weight`, 0.5 by default) is subtracted, and the result is kept at or above zero.

`ashrae241_group_mapping.csv` assigns each occupational group one ASHRAE-241 room. Volume, occupants, baseline outdoor airflow and the ASHRAE-241 eCADR come from `ashrae241_eca_by_occupancy.csv` (ASHRAE 241-2023 Table 5-1; representative volumes and occupants from Jones et al. 2025, Table 4). Indoor essential workers and indoor vital workers in a group are each multiplied by that room's net eCADR per person. A country's requirement is the sum of its groups. Countries without an ILO occupation breakdown take the mean of similar countries.

The COVID requirement columns use `covid_qer_ratio` (1, so the viral-load scale-up is left out) and `covid_u_new_other` (0). Health care still uses `u_new_healthcare`. `ashrae241_scaled_table1.csv` is the scaled rate before the outdoor-air credit. The `Scaled ECA` column below is the rate after it, which is what the worker requirements use. `ashrae241_scaled_by_mask_efficiency.csv` repeats the scale-up at each efficiency in `mask_efficiencies`.


In [ ]:
import ecadr_requirements as ecadr

rooms = ecadr.load_room_types()
print(
    f"Quanta emission ratio at the {parameters['viral_load_percentile']}th percentile "
    f"({parameters['ashrae_pathogen']}, {parameters['viral_load_sd_mode']} SD): "
    f"{ecadr.qer_ratio(parameters):.2f}"
)
display(results['ashrae_table'])
net = (
    results['by_group'][[
        'occupational_group', 'occupancy_group', 'occupancy_category',
        'Scaled ECA (L/s/person)',
    ]]
    .drop_duplicates('occupational_group')
    .set_index('occupational_group')
    .loc[rooms['occupational_group']]
)
display(net)


## 7. On-site housing worker requirements

ISCO **61** (market-oriented skilled agricultural) and **63** (subsistence farmers)
are treated as already on-site; they are subtracted from housing-relevant totals
(weighted by each series' total-weight column).


In [ ]:
housing = results['onsite_housing']
display(housing.head(10))


## 8. Validation — model vs calibrated vs ILO

`python src/essential_workers_validation.py` writes this table to
`results/essential_workers/validation/essential_workers_validation.csv`.


In [ ]:
ilo_validation = validation.validate_against_ilo(results)
table = ilo_validation['table']
print(f"Mean |Δ| calibrated: {ilo_validation['mean_abs_delta_pp']:.2f} pp   "
      f"model: {table['Delta model (pp)'].abs().mean():.2f} pp")
print(f"Outliers > {parameters['ilo_outlier_threshold_pp']:.0f} pp: {len(ilo_validation['outliers'])}")
display(ilo_validation['outliers'][[
    'Country Name', 'Our %Essential (calibrated)', 'ILO %essential (published)',
    'Delta calibrated (pp)',
]].head(10))


### 8.1 Regional aggregates


In [ ]:
display(results['by_region'][[
    'Region', ew.LABOUR_FORCE_COL, 'Indoor Essential Workers',
    'Essential Workers', '%Indoor Essential Workers',
]].head(12))


## 9. Optional visualization

The manuscript figures come from `scripts/visualization/` and are written to
`results/essential_workers/visualizations/`:

```bash
python scripts/visualization/plot_essential_workers.py  # worker share maps
python scripts/visualization/plot_workers_vs_gdp.py     # worker shares against GDP
python scripts/visualization/plot_group_composition.py  # occupational composition
```

Quick static view of the top countries by % indoor essential:


In [ ]:
top = by_country.nlargest(15, '%Indoor Essential Workers')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top['Country Name'], top['%Indoor Essential Workers'] * 100)
ax.set_xlabel('% Indoor Essential Workers')
ax.set_title('Top 15 countries — % indoor essential (calibrated)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 10. Write outputs & re-run tests

The quick-start cell already wrote the core tables. To refresh everything
from the repository root:

```bash
python src/essential_workers.py
python src/essential_workers_validation.py
```

Tests (from the repository root):

```bash
pytest tests/test_essential_workers.py
```


In [ ]:
for path in sorted(RESULTS.glob('*.csv')) + sorted((RESULTS / 'validation').glob('*.csv')):
    print(f'  {path.relative_to(RESULTS)}  {path.stat().st_size:,} bytes')


## 11. Key results

Numbers for the essential and vital worker results section of the manuscript,
using the indoor method in `parameters_essential_workers.csv`.

Run section 1 first, or this cell re-runs the model if `results` is not in memory.


In [ ]:
# Re-run the model if this section is run on its own
if 'results' not in globals():
    results = ew.estimate()
lf = results['by_country']
ilo_validation = validation.validate_against_ilo(results)

global_summary = validation.global_worker_summary(lf)
range_compression = validation.indoor_range_compression(lf)
_, gdp_correlations = validation.worker_shares_vs_gdp(lf, results['by_group'])
group_composition = ew.group_composition(results['by_group'])
ranked = lf.assign(**{
    '% Essential': 100 * lf['%Essential Workers'],
    '% Vital': 100 * lf['%Vital Workers'],
})

labour_force = global_summary.attrs['labour_force']


def fmt_pct(value, digits=1):
    """Format a percentage for manuscript text."""
    return f"{value:.{digits}f}%"


def fmt_billion(count):
    """Format a worker count in billions."""
    return f"{count / 1e9:.2f}"

def fmt_million(count):
    """Format a worker count in millions."""
    return f"{count / 1e6:.0f}"

def fmt_p(p_value):
    """Format a p-value for manuscript text."""
    if p_value < 0.001:
        return "p < 0.001"
    return f"p = {p_value:.3f}"


def country_pct(code, column):
    """Return one country's share as a percent."""
    row = lf.loc[lf['Country Code'] == code, column]
    return float(row.iloc[0] * 100.0)


def correlation_row(column):
    """Look up one GDP correlation row."""
    match = gdp_correlations.loc[gdp_correlations['column'] == column].iloc[0]
    return float(match['Spearman ρ']), float(match['Spearman p'])


def sector_table(column):
    """Occupational-group shares sorted largest first."""
    table = group_composition[['occupational_group', column]].copy()
    table[column] = 100.0 * table[column]
    return table.sort_values(column, ascending=False)

print('=' * 72)
print('ESSENTIAL AND VITAL WORKERS')
print(f"Indoor context method: {parameters['indoor_context_method']}")
print('=' * 72)

essential_pct = global_summary.loc['Essential workers', '% of Labour Force']
essential_workers = global_summary.loc['Essential workers', 'Workers']
vital_pct = global_summary.loc['Vital workers', '% of Labour Force']
vital_workers = global_summary.loc['Vital workers', 'Workers']
indoor_essential_pct = global_summary.loc['Indoor essential workers', '% of Labour Force']
indoor_essential_workers = global_summary.loc['Indoor essential workers', 'Workers']
indoor_vital_pct = global_summary.loc['Indoor vital workers', '% of Labour Force']
indoor_vital_workers = global_summary.loc['Indoor vital workers', 'Workers']

print(
    f"\nEssential workers make up {fmt_pct(essential_pct)} of the labour force "
    f"across the globe, totalling {fmt_billion(essential_workers)} billion people "
    f"in {len(lf)} countries and territories."
)
print(
    f"These results match the ILO's finding of {ilo_validation['mean_ilo_pct_essential']:.0f}% "
    f"of the labour force across {len(ilo_validation['table'])} countries, with a mean "
    f"country-level difference of {ilo_validation['mean_abs_delta_pp']:.2f} percentage points."
)

essential_bottom = ranked.nsmallest(1, '% Essential').iloc[0]
essential_top_two = ranked.nlargest(2, '% Essential')
max_names = ' and '.join(essential_top_two['Country Name'])
max_range = (
    f"{essential_top_two['% Essential'].iloc[-1]:.1f}"
    f"–{essential_top_two['% Essential'].iloc[0]:.1f}%"
)
print(
    f"\nAt the country level, essential worker percentages vary from "
    f"{essential_bottom['% Essential']:.1f}% in {essential_bottom['Country Name']} "
    f"to {max_range} in countries such as {max_names}."
)

rho_essential, p_essential = correlation_row('%Essential Workers')
rho_vital, p_vital = correlation_row('%Vital Workers')
print(
    f"\nWealthier countries have fewer essential workers relative to their workforce "
    f"({fmt_p(p_essential)}; Spearman rho = {rho_essential:.2f}) and the same holds for "
    f"vital workers ({fmt_p(p_vital)}; Spearman rho = {rho_vital:.2f})."
)

print(
    f"\nOverall, vital workers make up {fmt_pct(vital_pct)} of the world's labour force, "
    f"equal to {fmt_billion(vital_workers)} billion people."
)
print(
    f"Luxembourg decreases from {country_pct('LUX', '%Essential Workers'):.2f}% essential "
    f"to {country_pct('LUX', '%Vital Workers'):.2f}% vital."
)
print(
    f"Mozambique decreases from {country_pct('MOZ', '%Essential Workers'):.1f}% essential "
    f"to {country_pct('MOZ', '%Vital Workers'):.1f}% vital."
)
vital_top = ranked.nlargest(1, '% Vital').iloc[0]
print(
    f"The largest vital workforce is in {vital_top['Country Name']} "
    f"at {vital_top['% Vital']:.1f}%."
)

print(
    f"\nGlobally, indoor essential workers make up {fmt_pct(indoor_essential_pct)} of the "
    f"labour force ({fmt_million(indoor_essential_workers)} million people) and indoor vital "
    f"workers make up {fmt_pct(indoor_vital_pct)} ({fmt_million(indoor_vital_workers)} million)."
)

rho_indoor_essential, p_indoor_essential = correlation_row('%Indoor Essential Workers')
rho_indoor_vital, p_indoor_vital = correlation_row('%Indoor Vital Workers')
print(
    f"Indoor essential shares are still higher in lower-income countries ({fmt_p(p_indoor_essential)}; "
    f"rho = {rho_indoor_essential:.2f}), as are indoor vital shares ({fmt_p(p_indoor_vital)}; "
    f"rho = {rho_indoor_vital:.2f})."
)
print(
    f"Spearman's rho drops from {rho_essential:.2f} to {rho_indoor_essential:.2f} for essential "
    f"workers and from {rho_vital:.2f} to {rho_indoor_vital:.2f} for vital workers."
)

for transition in range_compression.index:
    row = range_compression.loc[transition]
    print(
        f"\n{transition}: p (indoor SD < total) = {row['p (indoor SD < total)']:.2e}; "
        f"p (indoor log-SD < total) = {row['p (indoor log-SD < total)']:.2e}"
    )

print('\nSector composition (global, worker-weighted):')
print('\nEssential workers:')
for _, row in sector_table('% of Essential Workers').iterrows():
    print(f"  {row['occupational_group']}: {row['% of Essential Workers']:.1f}%")
print('\nVital workers:')
for _, row in sector_table('% of Vital Workers').iterrows():
    print(f"  {row['occupational_group']}: {row['% of Vital Workers']:.1f}%")

rho_food_essential, p_food_essential = correlation_row('Food % of Essential Workers')
rho_food_vital, p_food_vital = correlation_row('Food % of Vital Workers')
print(
    f"\nFood is proportionally larger for poorer countries (essential: {fmt_p(p_food_essential)}, "
    f"rho = {rho_food_essential:.2f}; vital: {fmt_p(p_food_vital)}, rho = {rho_food_vital:.2f})."
)

print('\n--- Tables for copy/paste ---')
display(global_summary)
display(range_compression)
display(gdp_correlations)
display(sector_table('% of Essential Workers').rename(columns={'% of Essential Workers': 'share %'}))
display(sector_table('% of Vital Workers').rename(columns={'% of Vital Workers': 'share %'}))
